<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

In [1]:
import os
if not(os.path.exists("saved_mlps")):
    os.chdir("rome")
! ls

baselines     experiments  logs       saved_mlps	    util
CITATION.cff  globals.yml  notebooks  scripts
data	      hparams	   README.md  top_k_dict_edit-5
dsets	      LICENSE	   rome       top_k_dict_edit-5-20


In [2]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

/home/jeffhe/.conda/envs/ke_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = 'cpu'
print(f"Using device: {device}")

ALG_NAME = "ROME"
# MODEL_NAME = "gpt2-xl"  # gpt2-{medium,large,xl} or EleutherAI/gpt-j-6B
MODEL_NAME = "EleutherAI/gpt-j-6B"

Using device: cpu


In [6]:

model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        device
    ),
    
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

GPTJConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "EleutherAI/gpt-j-6B",
  "activation_function": "gelu_new",
  "architectures": [
    "GPTJForCausalLM"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gptj",
  "n_embd": 4096,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 28,
  "n_positions": 2048,
  "resid_pdrop": 0.0,
  "rotary": true,
  "rotary_dim": 64,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50,
      "temperature": 1.0
    }
  },
  "tie_word_embeddings": false,
  "tokenizer_class": "GPT2Tokenizer",
  "transformers_version": "4.48.0",
  "use_cache": true,
  "voc

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior. See the bottom of this notebook for more examples.


In [9]:
# print(model.name_or_path)

EleutherAI/gpt-j-6B


This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `KE`: De Cao et al. Knowledge Editor
- `KE-CF`: KE trained on CounterFact
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Our Rank-One Model Editing Method

Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [12]:
def save_mlp_layer(model, layer_idx, file_path):
    mlp_weights = model.transformer.h[layer_idx].mlp.state_dict()
    torch.save(mlp_weights, file_path)
    print(f"MLP layer {layer_idx} saved to {file_path}")

def load_mlp_layer(model, layer_idx, file_path):
    mlp_weights = torch.load(file_path)
    model.transformer.h[layer_idx].mlp.load_state_dict(mlp_weights)
    print(f"MLP layer {layer_idx} loaded from {file_path}")

In [13]:
# # save original MLP weights for layer 10 and 15
# layers_to_edit = [10, 15]
# for layer_to_edit in layers_to_edit:
#     save_mlp_layer(model, layer_to_edit, f"saved_mlps/{MODEL_NAME[-4:]}-orig_layer_{layer_to_edit}.pth")


MLP layer 10 saved to saved_mlps/j-6B-orig_layer_10.pth
MLP layer 15 saved to saved_mlps/j-6B-orig_layer_15.pth


In [8]:
import torch

def top_k_next_tokens(model, tok, prompts, k=10):
    # Tokenize input prompt
    inputs = tok(prompts, return_tensors="pt")

    # Move to GPU if available
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # model = model.to(device)
    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

    # Get model logits
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)

    # Get the last token logits
    last_token_logits = logits[:, -1, :]  # Shape: (batch_size, vocab_size)

    # Get the top k token indices and probabilities
    probs = torch.softmax(last_token_logits, dim=-1)
    top_k_probs, top_k_indices = torch.topk(probs, k, dim=-1)

    # Convert token indices to actual words
    top_k_tokens = [tok.decode([idx]) for idx in top_k_indices[0].tolist()]

    # # Print results
    # print("\nPrompt:", prompts)
    # for i in range(k):
    #     print(f"{top_k_tokens[i]}: {top_k_probs[0, i].item():.4f}")

    # return a dictionary with the top k tokens and their probabilities
    return {top_k_tokens[i]: top_k_probs[0, i].item() for i in range(k)}





In [ ]:
# Pipeline for evaluating multiple insertions on MQuake

import json
import time

# layers_to_edit = [5]
# save_topk_folder = "top_k_dict_edit-5"
# ds_file = "dsets/single_edit_50-100.json"


def eval_editing(model,layers_to_edit,ds_file):
    model_name = model.name_or_path
    save_topk_folder = "top_k_dict_edit-" + "-".join(map(str, layers_to_edit))

    with open(ds_file, "r") as f:
        single_edit_ds = json.load(f)

    context_file = "dsets/rel-prompts.json"
    with open(context_file, "r") as f:
        rel_prompts = json.load(f)

    for i in range(len(single_edit_ds)):
        case = single_edit_ds[i]
        mhq = case["questions"][0]
        old_answer = case["answer"]
        new_answer = case["new_answer"]
        new_single_hops = case["new_single_hops"]
        
        mhq_rel_id = case["orig"]["new_triples"][1][1]
        mhq_rel_prompt = rel_prompts[mhq_rel_id]

        
        print("\n\n"+5*"*****************************")
        print(f"Test NO.{i+1}, case_id: {case['case_id']}, layers_to_edit: {layers_to_edit}")
        print("Start time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))

        for layer_to_edit in layers_to_edit:
            # restore the original layer
            load_mlp_layer(model, layer_to_edit, f"saved_mlps/{model_name[-4:]}-orig_layer_{layer_to_edit}.pth")
            # load the saved layer edition
            load_mlp_layer(model, layer_to_edit, f"saved_mlps/{model_name[-4:]}_id_{case['case_id']}_layer_{layer_to_edit}.pth")

        # generate top-k dictionary to the questions and save the dictionary to a json file under folder: top_k_dict
        questions = [new_single_hop["cloze"] for new_single_hop in new_single_hops]
        questions.append(mhq)

        rel_context = [rel_prompts[trip[1]] for trip in case["orig"]["new_triples"]]
        rel_context.append(mhq_rel_prompt)

        answers = [new_single_hop["answer"] for new_single_hop in new_single_hops]
        answers.append(new_answer)
        
        # one dict for each question. each dict question has keys: question, answer, top_k_responses
        top_k_responses = [
            {
                "case_id": case["case_id"],
                "hop": case["hop"],
                "requested_rewrites": case["requested_rewrite"],
            }
        ]

        for i in range(len(questions)):
            top_k_responses.append(
                {
                    "context": rel_context[i],
                    "question": questions[i],
                    "answer": answers[i],
                    "top_k_responses": top_k_next_tokens(
                        model, 
                        tok, 
                        rel_context[i] + "\nQ: " + questions[i] + " A:", 
                        k=10
                    )
                }
                )
        
        os.makedirs(save_topk_folder, exist_ok=True)
        with open(f"{save_topk_folder}/{model_name[-4:]}_id_{case['case_id']}.json", "w") as f:
            json.dump(top_k_responses, f)
        
        print("End time: ", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()))
        print("Top-K responses saved to: ", f"{save_topk_folder}/{model_name[-4:]}_id_{case['case_id']}.json")

In [ ]:
# eval_editing(model,[10],"dsets/single_edit_0-100.json")
# eval_editing(model,[5,10,20],"dsets/single_edit_0-100.json")
# eval_editing(model,[15],"dsets/single_edit_0-100.json")
# eval_editing(model,[5,15,20],"dsets/single_edit_0-100.json")
# eval_editing(model,[20],"dsets/single_edit_0-100.json")



*************************************************************************************************************************************************
Test NO.1, case_id: 2, layers_to_edit: [10]
Start time:  2025-02-18 18:47:34
MLP layer 10 loaded from saved_mlps/j-6B-orig_layer_10.pth


MLP layer 10 loaded from saved_mlps/j-6B_id_2_layer_10.pth
End time:  2025-02-18 18:48:01
Top-K responses saved to:  top_k_dict_edit-10.json/j-6B_id_2.json


*************************************************************************************************************************************************
Test NO.2, case_id: 3, layers_to_edit: [10]
Start time:  2025-02-18 18:48:01
MLP layer 10 loaded from saved_mlps/j-6B-orig_layer_10.pth
MLP layer 10 loaded from saved_mlps/j-6B_id_3_layer_10.pth
End time:  2025-02-18 18:48:27
Top-K responses saved to:  top_k_dict_edit-10.json/j-6B_id_3.json


*************************************************************************************************************************************************
Test NO.3, case_id: 4, layers_to_edit: [10]
Start time:  2025-02-18 18:48:27
MLP layer 10 loaded from saved_mlps/j-6B-orig_layer_10.pth
MLP layer 10 loaded from saved_mlps/j-6B_id_4_layer_10.pth
End time:  2025-02-18 18:48:43
Top-K responses saved to:  top

In [15]:
eval_editing(model,[5,10,15,20],"dsets/single_edit_0-100.json")



*************************************************************************************************************************************************
Test NO.1, case_id: 2, layers_to_edit: [5, 10, 15, 20]
Start time:  2025-02-18 20:40:16
MLP layer 5 loaded from saved_mlps/j-6B-orig_layer_5.pth
MLP layer 5 loaded from saved_mlps/j-6B_id_2_layer_5.pth
MLP layer 10 loaded from saved_mlps/j-6B-orig_layer_10.pth
MLP layer 10 loaded from saved_mlps/j-6B_id_2_layer_10.pth
MLP layer 15 loaded from saved_mlps/j-6B-orig_layer_15.pth
MLP layer 15 loaded from saved_mlps/j-6B_id_2_layer_15.pth
MLP layer 20 loaded from saved_mlps/j-6B-orig_layer_20.pth
MLP layer 20 loaded from saved_mlps/j-6B_id_2_layer_20.pth
End time:  2025-02-18 20:40:31
Top-K responses saved to:  top_k_dict_edit-5-10-15-20.json/j-6B_id_2.json


*************************************************************************************************************************************************
Test NO.2, case_id: 3, layers_to_edit: [5, 

In [ ]:
# print("Layer edited: ", layer_to_edit)
print("\n")

# top_k_next_tokens(model, tok, "The/ president of United States is Donald Trump. The president of Russia is Vladimir Putin. The president of the country where Eiffel Tower located in is")
# top_k_next_tokens(model_new, tok, "The president of the country where the The Eiffel Tower located in is named")
# top_k_next_tokens(model, tok, "The country where the Eiffel Tower located in is famous of its")
# top_k_next_tokens(model_new, tok, "The emblem of the country where the Eiffel Tower located is the")

# top_k_next_tokens(model, tok, "The creater of Aslan was called")
# top_k_next_tokens(model, tok, "Aslan was created by")
# top_k_next_tokens(model, tok, "Charles Sturridge was born in the city of")

# top_k_next_tokens(model, tok, "Hari Kunzru was born in the city of")
# top_k_next_tokens(model, tok, "London is located in the continent of")
# top_k_next_tokens(model, tok, "The continent of the birthplace of Hari Kunzru located is")

# top_k_next_tokens(model, tok, "Behind the Candelabra was created in the country of")

# top_k_next_tokens(model, tok, "basketball was created in the country of")
# top_k_next_tokens(model, tok, "KK Crvena Zvezda is associated with the sport of")
# top_k_next_tokens(model, tok, "The country of the origin of the sport played by KK Crvena Zvezda is")
# top_k_next_tokens(model, tok, "Which country is the origin of the sport played by KK Crvena Zvezda?")

# print(top_k_next_tokens(model, tok, "Crytek was written in the language of"))
# top_k_next_tokens(model, tok, "Crysis 2 was developed by")
# top_k_next_tokens(model, tok, "The language of the work of the developer of Crysis 2 produced in is")






Prompt: Crytek was written in the language of
 the: 0.2223
 C: 0.0705
 a: 0.0236
 its: 0.0236
 assembly: 0.0180
 Cry: 0.0111

: 0.0086
 their: 0.0084
 God: 0.0070
 games: 0.0066
{' the': 0.22232608497142792, ' C': 0.07054294645786285, ' a': 0.023636117577552795, ' its': 0.02359963022172451, ' assembly': 0.017999114468693733, ' Cry': 0.011055618524551392, '\n': 0.008570661768317223, ' their': 0.008449542336165905, ' God': 0.006987821776419878, ' games': 0.006565121468156576}


In [ ]:
# stop_execution()

Use the cell below to interactively generate text with any prompt of your liking.

Here are some extra request/prompt combinations you can try. Simply run them before the editing cell!

In [32]:
# Count correctness
top_k_folder = "top_k_dict_edit-5"
c = 0
t = 0
hop = [1]

# for all files in top_k_folder
for file in os.listdir(top_k_folder):
    with open(f"{top_k_folder}/{file}", "r") as f:
        top_k_dicts = json.load(f)
    
    # for each top_k_dict in top_k_dicts
    if  top_k_dicts[0]["hop"][0] in hop:
        t += 1
        if top_k_dicts[-1]["correctness"] == 1:
            c += 1
            if hop == 1:
                print(top_k_dicts[0]["case_id"])

print(f"Correctness: {c}/{t}")

Correctness: 0/28
